# Chatbot de recomendação de livros

O chatbot não aprende nada novo. Ele só traduz uma frase em português para uma chamada do `RecomendadorLivros` que vocês já fizeram. Toda a inteligência de recomendação (TF-IDF, BERT, fusão RRF, exclusão de outras edições) continua na classe do outro notebook. O chatbot faz o trabalho de entender o que o usuário pediu e de apresentar a resposta.

## O caminho de uma mensagem

Tome como exemplo `"quero um livro sobre guerra e política"`. O método `responder()` faz isto, nesta ordem:

**1. Normaliza o texto.** A função `normalizar` passa tudo para minúsculas, remove acentos (via `unicodedata.normalize("NFKD")`, a mesma técnica que vocês usam em `normalizar_para_regra`) e colapsa espaços. Assim "Política", "politica" e "POLÍTICA" viram a mesma coisa para as regex.

**2. Checa comandos fixos.** Se a mensagem for "sair", "ajuda", uma saudação, "mais" ou "por que", ela é tratada na hora e não chega ao recomendador.

**3. Procura um gênero na frase.** `achar_genero` percorre o dicionário `GENEROS`, que liga palavras em português aos rótulos que a `identificar_genero` do notebook de vocês gera. "guerra" vira `"Guerra"`, "terror" vira `"Terror/gótico"`, e assim por diante. As chaves são testadas da mais longa para a mais curta, para que "ficção científica" ganhe de "ficção". Se achar, o gênero vira o parâmetro `genero=` do recomendador, que já sabe filtrar o pool.

**4. Decide a intenção.** Aqui entra uma regex (`PADRAO_SIMILAR`) que procura construções como "parecido com X", "gostei de X" e "similar a X":

- Se a regex casar, o que vem depois é a lista de livros. `separar_livros` quebra por vírgula e "e". Com um livro, chama `recomendar_similares`. Com vários, chama `recomendar_por_perfil`.
- Se não casar, a frase inteira é tratada como um tema e vai para `recomendar_por_consulta`.

**5. Prepara a consulta (só no caso de tema).** O corpus e o MiniLM são em inglês, então `traduzir` troca palavra por palavra usando o `GLOSSARIO` ("guerra" vira "war", "política" vira "politics"). Depois removemos palavras de comando como "quero", "livro" e "sobre", que só poluiriam o TF-IDF. A consulta final fica `"war politics"`, a mesma do notebook de vocês.

**6. Chama o recomendador e formata.** O resultado é um DataFrame. `_formatar` o transforma em lista numerada com título, autor e gênero.

## Por que os três tipos de recomendação mapeiam tão bem

Eles já existiam no notebook de vocês, com entradas diferentes:

| O usuário diz | Método chamado | O que entra |
|---|---|---|
| um tema | `recomendar_por_consulta` | texto livre |
| "parecido com Dracula" | `recomendar_similares` | um livro |
| "gostei de A, B e C" | `recomendar_por_perfil` | vários livros (média dos vetores) |

Todos usam `modelo=["tfidf","bert"]`, o híbrido com RRF que vocês definiram como `HIBRIDO`.

## A memória da conversa

A classe `Estado` guarda o último resultado e os argumentos da última chamada. Isso permite duas coisas:

- **"mais"**: repete a mesma chamada pedindo `n + 5` resultados e mostra só os novos (`.iloc[c["n"]:]`). É um jeito simples de paginar sem mudar o recomendador.
- **"por que"**: lê a coluna `termos_em_comum` do último resultado. Essa coluna vem do TF-IDF e lista os termos que mais contribuíram para a similaridade. Quando ela está vazia, o bot diz que a semelhança foi semântica (BERT).

## O que o chatbot não faz

- **Não tem NLU de verdade.** A detecção de intenção é por regex. Frases fora dos padrões ("tem algo no estilo de Dracula?") caem em "consulta" e provavelmente dão um resultado ruim.
- **O glossário é pequeno.** Palavra que não está nele vai em português para o TF-IDF, que não a encontra no vocabulário. O BERT ajuda um pouco, mas o MiniLM usado é só em inglês.
- **O filtro de gênero é automático.** Dizer "guerra" restringe a busca ao gênero Guerra, mesmo quando a pessoa queria só o tema. Dá para mudar isso se o grupo achar melhor.
- **Títulos precisam estar em inglês**, como no Gutenberg. O `localizar()` já faz busca aproximada e sugere o mais parecido quando há erro de digitação.

## Como melhorar depois

O ponto mais fraco é a detecção de intenção por regex. Como vocês já têm um notebook de classificação, a evolução natural é treinar um classificador (TF-IDF + regressão logística) com algumas dezenas de frases de exemplo para cada intenção (tema, similar, perfil, mais, explicar). Isso encaixa bem no escopo da matéria e substitui as regex por algo medido, com acurácia que vocês podem reportar.

Se quiser, posso detalhar alguma parte, como a regex de similaridade ou a lógica do "mais", ou fazer essa versão com classificador de intenção.


## 1. Carregar o recomendador

Executa só as células de definição do `recomendacao_livros.ipynb` (imports, regras de gênero, catálogo e a classe), sem rodar as demonstrações e a avaliação.

In [1]:
import json

with open("recomendacao_livros.ipynb", encoding="utf-8") as f:
    _nb = json.load(f)
_celulas = ["".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code"]

for _c in _celulas[:4]:
    exec(_c)

print("Classe carregada:", RecomendadorLivros.__name__)

Classe carregada: RecomendadorLivros


## 2. Normaliza o texto e tenta encontrar o genero na frase

In [2]:
import re
import unicodedata
from dataclasses import dataclass, field
from typing import Optional

HIBRIDO = ["tfidf", "bert"]


def normalizar(txt: str) -> str:
    txt = unicodedata.normalize("NFKD", txt.lower())
    txt = "".join(c for c in txt if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", txt).strip()

GLOSSARIO = {
    "guerra": "war", "amor": "love", "casamento": "marriage", "morte": "death",
    "mar": "sea", "viagem": "journey", "futuro": "future", "espaco": "space",
    "fantasma": "ghost", "vampiro": "vampire", "monstro": "monster",
    "crime": "crime", "detetive": "detective", "assassinato": "murder",
    "familia": "family", "politica": "politics", "religiao": "religion",
    "pirata": "pirate", "navio": "ship", "ciencia": "science", "rei": "king",
    "rainha": "queen", "aventura": "adventure", "misterio": "mystery",
    "terror": "horror", "romance": "romance", "poesia": "poetry",
    "infantil": "children", "escravidao": "slavery", "sociedade": "society",
}

GENEROS = {
    "ficcao cientifica": "Ficção científica/fantasia", "fantasia": "Ficção científica/fantasia",
    "misterio": "Mistério/policial", "policial": "Mistério/policial", "detetive": "Mistério/policial",
    "terror": "Terror/gótico", "gotico": "Terror/gótico", "horror": "Terror/gótico",
    "romance": "Romance", "guerra": "Guerra", "aventura": "Aventura",
    "ficcao historica": "Ficção histórica", "faroeste": "Faroeste",
    "humor": "Humor", "poesia": "Poesia", "infantil": "Literatura infantil/juvenil",
    "juvenil": "Literatura infantil/juvenil",
}


def traduzir(txt: str) -> str:
    return " ".join(GLOSSARIO.get(p, p) for p in normalizar(txt).split())


def achar_genero(txt_norm: str) -> Optional[str]:
    for chave in sorted(GENEROS, key=len, reverse=True):
        if re.search(rf"\b{chave}\b", txt_norm):
            return GENEROS[chave]
    return None

## 3. Detecção de intenção

In [3]:
PADRAO_SAIR = r"^(sair|tchau|fim|quit|exit|ate mais)\b"
PADRAO_AJUDA = r"^(ajuda|help|o que voce faz|como funciona|\?)$"
PADRAO_SAUDACAO = r"^(oi|ola|bom dia|boa tarde|boa noite|e ai|hey|hi|hello)\b"
PADRAO_MAIS = r"^(mais|mais um|outros|me mostra mais|quero mais)\b"
PADRAO_PORQUE = r"^(por ?que|porque|pq|explica|por que esses)"
PADRAO_SIMILAR = r"(parecid\w+ com|similar\w* (a|ao|com)|gostei d[eoa]s?|li e gostei|curti|recomend\w+ (algo|livros?) (como|tipo))\s+(?P<livros>.+)"
PADRAO_PERFIL = r"(?P<livros>.+?)\s*(?:,| e )\s*(?P<resto>.+)"


def separar_livros(trecho: str) -> list:
    trecho = re.sub(r"[\"“”']", "", trecho)
    partes = re.split(r"\s*(?:,|;|\be\b|\band\b)\s*", trecho)
    return [p.strip(" .!?") for p in partes if p.strip(" .!?")]

## 4. O chatbot


In [4]:
@dataclass
class Estado:
    """Memória curta da conversa."""
    ultimo_resultado: object = None
    ultima_chamada: dict = field(default_factory=dict)
    top_n: int = 5


class ChatbotLivros:
    def __init__(self, rec: RecomendadorLivros):
        self.rec = rec
        self.estado = Estado()

    def responder(self, mensagem: str) -> str:
        bruto = mensagem.strip()
        msg = normalizar(bruto)
        if not msg:
            return "Pode me dizer o que você quer ler? Digite 'ajuda' para ver exemplos."

        if re.search(PADRAO_SAIR, msg):
            return "__SAIR__"
        if re.match(PADRAO_AJUDA, msg):
            return self.ajuda()
        if re.match(PADRAO_SAUDACAO, msg) and len(msg.split()) <= 3:
            return "Olá! Sou o chatbot de livros do Project Gutenberg. " \
                   "Me conte um tema, um gênero ou livros que você gostou. ('ajuda' mostra exemplos)"
        if re.match(PADRAO_MAIS, msg):
            return self.mais()
        if re.match(PADRAO_PORQUE, msg):
            return self.explicar()

        genero = achar_genero(msg)

        m = re.search(PADRAO_SIMILAR, bruto, flags=re.I)
        if m:
            livros = separar_livros(m.group("livros"))
            return self.por_livros(livros, genero)

        return self.por_consulta(bruto, genero)

    def por_consulta(self, texto: str, genero: Optional[str]) -> str:
        consulta = traduzir(texto)
        consulta = re.sub(r"\b(quero|livros?|sobre|de|com|que|tenha|me|indica|recomende|"
                          r"recomenda|algo|um|uma|sugira|sugere|gosto)\b", " ", consulta)
        consulta = re.sub(r"\s+", " ", consulta).strip()
        if not consulta:
            return "Não entendi o tema. Tente algo como: 'quero um livro sobre guerra e política'."
        return self._rodar("consulta", {"consulta": consulta, "genero": genero}, texto)

    def por_livros(self, livros: list, genero: Optional[str]) -> str:
        try:
            if len(livros) == 1:
                return self._rodar("similares", {"livro": livros[0], "genero": genero}, livros[0])
            return self._rodar("perfil", {"livros": livros, "genero": genero}, ", ".join(livros))
        except ValueError as e:
            return f"{e}\nTente o título em inglês, como está no Gutenberg."

    def _rodar(self, tipo: str, args: dict, rotulo: str, top_n: Optional[int] = None) -> str:
        n = top_n or self.estado.top_n
        modelo = HIBRIDO if all(m in self.rec.modelos for m in HIBRIDO) else "hibrido"
        if tipo == "consulta":
            df = self.rec.recomendar_por_consulta(args["consulta"], n, modelo, args["genero"])
        elif tipo == "similares":
            df = self.rec.recomendar_similares(args["livro"], n, modelo, args["genero"])
        else:
            df = self.rec.recomendar_por_perfil(args["livros"], n, modelo, args["genero"])

        self.estado.ultimo_resultado = df
        self.estado.ultima_chamada = {"tipo": tipo, "args": args, "n": n}
        if df.empty:
            return "Não encontrei nada relevante. Tente outras palavras ou remova o filtro de gênero."
        return self._formatar(df, rotulo, args.get("genero"))

    def mais(self) -> str:
        c = self.estado.ultima_chamada
        if not c:
            return "Ainda não recomendei nada. Me diga um tema ou um livro que você gostou."
        n = c["n"] + self.estado.top_n
        df = self._chamar(c, n).iloc[c["n"]:]
        if df.empty:
            return "Acabaram as sugestões para essa busca."
        c["n"] = n
        self.estado.ultimo_resultado = df
        return self._formatar(df, "mais resultados", c["args"].get("genero"), cabecalho=False)

    def _chamar(self, c, n):
        modelo = HIBRIDO if all(m in self.rec.modelos for m in HIBRIDO) else "hibrido"
        a = c["args"]
        if c["tipo"] == "consulta":
            return self.rec.recomendar_por_consulta(a["consulta"], n, modelo, a["genero"])
        if c["tipo"] == "similares":
            return self.rec.recomendar_similares(a["livro"], n, modelo, a["genero"])
        return self.rec.recomendar_por_perfil(a["livros"], n, modelo, a["genero"])

    def explicar(self) -> str:
        df = self.estado.ultimo_resultado
        if df is None or df.empty:
            return "Ainda não tenho uma recomendação para explicar."
        linhas = ["Os termos que mais pesaram no TF-IDF para cada sugestão:"]
        for _, r in df.iterrows():
            termos = r["termos_em_comum"] or "(semelhança semântica via BERT, sem termos em comum)"
            linhas.append(f"  • {r['title']}: {termos}")
        return "\n".join(linhas)

    @staticmethod
    def _formatar(df, rotulo, genero, cabecalho=True) -> str:
        linhas = []
        if cabecalho:
            extra = f" (gênero: {genero})" if genero else ""
            linhas.append(f"Sugestões para “{rotulo}”{extra}:")
        for i, r in enumerate(df.itertuples(), 1):
            gen = f" [{r.genero}]" if isinstance(r.genero, str) else ""
            linhas.append(f"{i}. {r.title} — {r.autor}{gen}")
        linhas.append("Digite 'mais', 'por que' ou faça outra pergunta.")
        return "\n".join(linhas)

    @staticmethod
    def ajuda() -> str:
        return (
            "Posso recomendar de três formas:\n"
            "  • Por tema:   'quero um livro sobre guerra e política'\n"
            "  • Por gênero: 'romance sobre casamento' / 'terror com fantasmas'\n"
            "  • Por livro:  'parecido com Dracula'\n"
            "  • Por perfil: 'gostei de Dracula, Frankenstein e Jekyll'\n"
            "Depois: 'mais' (próximas sugestões), 'por que' (explica), 'sair'."
        )

## 5. Instanciar (carrega catálogo + BERT, pode demorar na 1ª vez)

In [5]:
df = carregar_catalogo("gutenberg_books.csv")
rec = RecomendadorLivros(df, modelos=("tfidf", "bert"))
bot = ChatbotLivros(rec)
print(bot.ajuda())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Posso recomendar de três formas:
  • Por tema:   'quero um livro sobre guerra e política'
  • Por gênero: 'romance sobre casamento' / 'terror com fantasmas'
  • Por livro:  'parecido com Dracula'
  • Por perfil: 'gostei de Dracula, Frankenstein e Jekyll'
Depois: 'mais' (próximas sugestões), 'por que' (explica), 'sair'.


## 6. Demonstração (conversa de exemplo)

In [6]:
for msg in [
    "oi",
    "quero um livro sobre guerra e política",
    "por que",
    "parecido com Dracula",
    "gostei de Dracula, Frankenstein e Jekyll",
    "romance sobre casamento",
    "mais",
]:
    print("você>", msg)
    print("bot>", bot.responder(msg), "\n")

você> oi
bot> Olá! Sou o chatbot de livros do Project Gutenberg. Me conte um tema, um gênero ou livros que você gostou. ('ajuda' mostra exemplos) 

você> quero um livro sobre guerra e política
bot> Sugestões para “quero um livro sobre guerra e política” (gênero: Guerra):
1. War and Peace — Leo Tolstoy [Guerra]
2. The Red Badge of Courage: An Episode of the American Civil War — Stephen Crane [Guerra]
3. The Zeppelin's Passenger — E. Phillips (Edward Phillips) Oppenheim [Guerra]
4. A farewell to arms — Ernest Hemingway [Guerra]
5. The Girl Philippa —  [Guerra]
Digite 'mais', 'por que' ou faça outra pergunta. 

você> por que
bot> Os termos que mais pesaram no TF-IDF para cada sugestão:
  • War and Peace: war
  • The Red Badge of Courage: An Episode of the American Civil War: war
  • The Zeppelin's Passenger: war
  • A farewell to arms: war
  • The Girl Philippa: war 

você> parecido com Dracula
bot> Sugestões para “Dracula”:
1. Carmilla — Joseph Sheridan Le Fanu
2. Frankenstein; Or, The M

## 7. Conversa interativa

Execute a célula e digite no campo que aparece. Digite `sair` para encerrar.

In [8]:
while True:
    msg = input("você> ")
    resp = bot.responder(msg)
    if resp == "__SAIR__":
        print("bot> Até a próxima leitura!")
        break
    print("bot>", resp, "\n")

você> guerra e paz
bot> Sugestões para “guerra e paz” (gênero: Guerra):
1. War and Peace — Leo Tolstoy [Guerra]
2. A farewell to arms — Ernest Hemingway [Guerra]
3. The Red Badge of Courage: An Episode of the American Civil War — Stephen Crane [Guerra]
4. The Girl Philippa —  [Guerra]
5. That Which Hath Wings: A Novel of the Day —  [Guerra]
Digite 'mais', 'por que' ou faça outra pergunta. 

você> ronaldo
bot> Sugestões para “ronaldo”:
1. Hypnerotomachia: The Strife of Loue in a Dreame — Francesco Colonna [Romance literário]
2. Camille (La Dame aux Camilias) — Alexandre Dumas [Romance]
3. Carmen — Prosper Mérimée
4. Don Quixote — Miguel de Cervantes Saavedra [Romance literário]
5. The Man Who Did the Right Thing: A Romance — 
Digite 'mais', 'por que' ou faça outra pergunta. 

você> tomar no cu
bot> Sugestões para “tomar no cu”:
1. The Adventures of Tom Sawyer, Complete — Mark Twain [Aventura]
2. Lorna Doone: A Romance of Exmoor — R. D. (Richard Doddridge) Blackmore [Romance]
3. Adventur